# Pritam — Day 3 Session Summary
**Sprint:** Dark Store + Integrated Logistics | **Date:** April 4, 2026 (Day 3 tasks completed on Day 4)  
**Branch:** `pritam_temp_apr1` | **Role:** Lead Coder + Integration Architect

---

**Why Day 3 ran on Day 4:**  
Pranav's `vrp_nodes.csv` was not delivered by Day 2 EOD. However, `forward_vrp.py` (committed by the team) builds VRP nodes internally from `master_df_v2.parquet` + `dark_stores_final.csv`, making the static hand-off file redundant. All Day 3 tasks were completed in full today.

Load [`docs/pritam_sess_summ.md`](../../docs/pritam_sess_summ.md) for Pritam-scoped context.  
Load [`session_summary_v3.md`](../../session_summary_v3.md) for full project context.

---
## 1. Context Sync — What the Team Had Committed

Before starting Day 3, `git merge main` pulled 22 new team commits into `pritam_temp_apr1`:

| New file | Owner | Purpose |
|----------|-------|---------|
| `src/forward_vrp.py` | Team | OR-Tools CVRPTW full implementation |
| `src/reverse_vrp.py` | Vybhav | Reverse pickup VRP |
| `src/route_parser.py` | Pranav | Shared VRP utilities, constants, node builder |
| `src/demand_forecasting.py` | Anurag | Prophet per-zone forecasting |
| `notebooks/03_forward_vrp.ipynb` | Team | Forward VRP notebook |
| `notebooks/05_reverse_vrp.ipynb` | Vybhav | Reverse VRP notebook |
| `notebooks/vrp_summary.ipynb` | Team | VRP consolidated summary |

### Key design resolution — Pranav's `vrp_nodes.csv`
The roadmap specified Pranav would deliver `vrp_nodes.csv` as a static file.  
The team instead implemented `build_vrp_nodes()` inside `route_parser.py` which builds nodes dynamically from `master_df_v2.parquet` + `dark_stores_final.csv`.  
**This is better engineering** — no fragile hand-off, nodes always consistent with the data.

---
## 2. Step 1 — Clustering Pipeline → `dark_stores_final.csv`

### What `clustering.run_full_pipeline()` does
```
1. load_sp_data()           — load master_df.parquet, filter to SP Metro (46% of SP state)
2. build_zip_level_coords() — aggregate to zip-code level; demand_weight = order_count
3. run_kmeans()             — sweep K ∈ {3..12}; record inertia + silhouette per K
4. pick_k_by_coverage()     — choose smallest K with ≥70% customers within 5 km
5. assign_voronoi()         — assign each customer to nearest centroid → dark_store_id
6. build_dark_stores_df()   — centroids table with capacity + zone KPIs
7. save_outputs()           — dark_stores_final.csv, master_df_v2.parquet, CSVs
```

### K selection — coverage rule overrides silhouette

$$\text{Optimal } K = \min\{K : \text{coverage}(K,\; 5\text{ km}) \geq 70\%\}$$

| K | Silhouette | Coverage (5 km) | Selected? |
|---|-----------|----------------|-----------|
| 3 | **0.369** | 24.2% | |
| 7 | 0.359 | 51.7% | |
| 10 | 0.359 | 68.6% | |
| **11** | 0.354 | **73.7%** | ✅ chosen |
| 12 | 0.346 | 78.0% | |

Silhouette preferred K=3 but that leaves 76% of customers >5 km — unworkable for dark store last-mile delivery.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src.clustering import run_full_pipeline as cluster_pipeline
import pandas as pd

results = cluster_pipeline(
    parquet_path="../../data/master_df.parquet",
    out_dir="../../data",
    plot_dir="../../outputs",
)

print(f"\nOptimal K    : {results['optimal_k']}")
print(f"Coverage 5km : {results['coverage_overall']*100:.1f}%")
results["dark_stores_df"][["dark_store_id","lat","lon","n_orders","coverage_5km_pct"]]